In [5]:
import shapely.geometry as sg

import json

import ee 
import geemap
import geopandas as gpd
import pandas as pd
import datetime as dt
import pprint as pp
from shapely.geometry import shape

ee.Authenticate()
ee.Initialize(project='ee-green-by-another-name')

In [7]:
area_data = pd.read_csv('./data/area_data_v1.csv')
area_data = area_data[area_data['level'] == 'toa']
area_data = area_data[['date', 'roi', 'ls_s2_percent_diff']]

area_data_dirty = area_data[
    (area_data['ls_s2_percent_diff'] < -25) |
    (area_data['ls_s2_percent_diff'] > 25)
]

area_data_dirty.head(20)

,date,roi,ls_s2_percent_diff
67,2022-07-01,MRD_sub1,46.365340
80,2019-07-11,AKCP_sub2,-53.494203
113,2016-06-11,AKCP_sub1,63.713683
116,2021-06-16,AKCP_sub1,46.459507


In [12]:
roi_name = 'AKCP_sub2'
date = '2019-07-11'

In [13]:
band_dict = {'Sentinel2_toa': ['B2', 
                              'B3',
                              'B4',
                              'B8'],
            'LandSat8_toa': ['B2',
                             'B3',
                             'B4',
                             'B5']
}

image_footprints_path = f'./data/overlap_dates_for_roi/{roi_name}_overlap_dates.shp'
best_image_dates = gpd.read_file(image_footprints_path) 
est_utm = f'EPSG:{best_image_dates.estimate_utm_crs().to_epsg()}' # Have to convert pyproj object into literal string for ee
geom = best_image_dates.geometry.iloc[0]
coords = geom.exterior.coords
coords_list = [[x, y] for x, y in coords]

In [14]:
asset_string = 'LANDSAT/LC08/C02/T1_TOA'
start_date = ee.Date(date)
# Advance one day so that the filterDate covers the 24‐hour period of that date.
end_date = start_date.advance(1, 'day')
roi = ee.Geometry.Polygon(coords_list)

# Filter the collection by date and location.
collection = ee.ImageCollection(asset_string) \
    .filterDate(start_date, end_date) \
    .filterBounds(roi)

mosaic = collection.mosaic().clip(roi)
first_img = ee.Image(collection.first())

ssa_band = mosaic.select('SAA')

stats = ssa_band.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=roi,
    scale=30,          # Use an appropriate scale for your dataset
    maxPixels=1e13
)

# Convert to a Python dictionary
stats_dict = stats.getInfo()

print(stats_dict)

sza_band = mosaic.select('SZA')

stats = sza_band.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=roi,
    scale=30,          # Use an appropriate scale for your dataset
    maxPixels=1e13
)

# Convert to a Python dictionary
stats_dict = stats.getInfo()

print(stats_dict)

pp.pp(first_img.get('SUN_AZIMUTH').getInfo())



{'SAA_max': 17462, 'SAA_min': -4778}
{'SZA_max': 8146, 'SZA_min': 4848}
175.39231963


In [15]:
asset_string = "COPERNICUS/S2_HARMONIZED"

# Filter the collection by date and location.
collection = ee.ImageCollection(asset_string) \
    .filterDate(start_date, end_date) \
    .filterBounds(roi)

mosaic = collection.mosaic().clip(roi)
first_img = ee.Image(collection.first())

print(first_img.get('MEAN_SOLAR_AZIMUTH_ANGLE').getInfo())
print(first_img.get('MEAN_SOLAR_ZENITH_ANGLE').getInfo())

183.173441073
47.7967558439


In [56]:
# True-color composite: Landsat 8 bands B4 (red), B3 (green), and B2 (blue).
rgb_vis = {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0,
    'max': 0.3,
}


# For visualizing angle properties, create constant images.
ssa_angle_vis = {
    'min': 17007,
    'max': 17327,
    'palette': ['blue', 'green', 'red']
}

sza_angle_vis = {
    'min': 5248,
    'max': 5295,
    'palette': ['blue', 'green', 'red']
}


In [50]:

Map = geemap.Map(zoom=8)

# Add the Landsat‑8 mosaic layer.
Map.addLayer(mosaic, rgb_vis, 'Landsat-8 Mosaic RGB')
Map.addLayer(ssa_band, ssa_angle_vis, "SAA")
Map.addLayer(sza_band, sza_angle_vis, "SZA")
Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…